In [ ]:
# JSON 파일을 읽어 VectorDB에 저장 후 검색
!pip install chromadb sentence-transformers

In [ ]:
import json
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient

DB_PATH = "./cource_db"
COLLECTION_NAME = "cources"

class MYCourceVectorDB:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.client = PersistentClient(path=DB_PATH)
        self.collection = (self.client.get_or_create_collection(COLLECTION_NAME))

    # json 적재
    def load_cources(self, json_file):
        with open(json_file, 'r', encoding='utf-8') as f:
            cources = json.load(f)

        ids = []
        docs = []
        metas = []

        for course in cources:
            document = f"""
                강의명:{course["course_name"]}
                카테고리:{course["category"]}
                난이도:{course["level"]}
                설명:{course["description"]}
                태그:{' '.join(course["tags"])}
            """
            ids.append(course['course_id'])
            docs.append(document)
            metas.append(
                {
                    "course_name":course["course_name"],
                    "instructor":course["instructor"],
                    "category":course["category"],
                    "level":course["level"]
                }
            )

        embeddings = self.model.encode(docs, normalize_embeddings=True).tolist()

        self.collection.upsert(
            ids = ids,
            documents=docs,
            metadatas = metas,
            embeddings=embeddings
        )
        print(f'{len(ids)}개 강의 저장 완료')


    # 강의 검색
    def search_cource(self, query, top_k=3):
        query_embedding = (self.model.encode([query], normalize_embeddings=True).tolist())
        result = self.collection.query(query_embeddings=query_embedding, n_results=top_k)
        print('\n검색 결과')
        for i in range(len(result['ids'][0])):
            meta = (result['metadatas'][0][i])
            print(f"""
                [{i + 1}]
                강의명 : {meta['course_name']}
                강사 : {meta['instructor']}
                카테고리 : {meta['category']}
                난이도 : {meta['level']}
                거리 : {result['distances'][0][i]:.4f}
            """)

    # 카테고리 필터 검색
    def search_by_category(self, query, category):
        emb = self.model.encode([query], normalize_embeddings=True).tolist()
        result = self.collection.query(
            query_embeddings=emb,
            n_results=3,
            where={'category':category}
        )
        print(f'\n{category} 검색')
        for meta in (result['metadatas'][0]):
            print(meta['course_name'])


    # 유사 강의 추천
    def recommaned_cource(self, course_name):
        pass


if __name__ == "__main__":
    dbobj = MYCourceVectorDB()
    dbobj.load_cources("courses.json")

    dbobj.search_cource("파이썬으로 웹 개발 배우고 싶어요")
    dbobj.search_by_category(query="파이썬", category="Programming")
    dbobj.recommaned_cource("Python 기초")
